# Deploying Iris-detection model using Vertex AI


## Overview

In this tutorial, you build a scikit-learn model and deploy it on Vertex AI using the custom container method. You use the FastAPI Python web server framework to create a prediction endpoint. You also incorporate a preprocessor from training pipeline into your online serving application.

Learn more about [Custom training](https://cloud.google.com/vertex-ai/docs/training/custom-training) and [Vertex AI Prediction](https://cloud.google.com/vertex-ai/docs/predictions/get-predictions).

### Objective

In this notebook, you learn how to create, deploy and serve a custom classification model on Vertex AI. This notebook focuses more on deploying the model than on the design of the model itself.


This tutorial uses the following Vertex AI services and resources:

- Vertex AI models
- Vertex AI endpoints

The steps performed include:

- Train a model that uses flower's measurements as input to predict the class of iris.
- Save the model and its serialized pre-processor.
- Build a FastAPI server to handle predictions and health checks.
- Build a custom container with model artifacts.
- Upload and deploy custom container to Vertex AI Endpoints.

### Dataset

This tutorial uses R.A. Fisher's Iris dataset, a small and popular dataset for machine learning experiments. Each instance has four numerical features, which are different measurements of a flower, and a target label that
categorizes the flower into: **Iris setosa**, **Iris versicolour** and **Iris virginica**.

This tutorial uses [a version of the Iris dataset available in the
scikit-learn library](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html#sklearn.datasets.load_iris).

### Costs

This tutorial uses billable components of Google Cloud:

* Vertex AI
* Cloud Storage
* Artifact Registry
* Cloud Build

Learn about [Vertex AI
pricing](https://cloud.google.com/vertex-ai/pricing), [Cloud Storage
pricing](https://cloud.google.com/storage/pricing), [Artifact Registry pricing](https://cloud.google.com/artifact-registry/pricing) and [Cloud Build pricing](https://cloud.google.com/build/pricing) and use the [Pricing
Calculator](https://cloud.google.com/products/calculator/)
to generate a cost estimate based on your projected usage.

## Get started

### Install Vertex AI SDK for Python and other required packages



In [1]:
!pip3 install --upgrade --quiet google-cloud-aiplatform "numpy>=2.0.2" --force-reinstall pandas scikit-learn scipy matplotlib

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow-skinny 3.18.0 requires opentelemetry-sdk<3,>=1.9.0, which is not installed.
opentelemetry-exporter-prometheus 0.66b1 requires opentelemetry-sdk~=1.45.1, which is not installed.
mlflow-tracing 3.18.0 requires opentelemetry-sdk<3,>=1.9.0, which is not installed.
kfp-pipeline-spec 2.17.3 requires protobuf<7.0,>=6.31.1, but you have protobuf 7.36.2 which is incompatible.
pandas-profiling 3.2.0 requires joblib~=1.1.0, but you have joblib 1.6.0 which is incompatible.
pyopenssl 24.0.0 requires cryptography<43,>=41.0.5, but you have cryptography 50.0.2 which is incompatible.
opentelemetry-semantic-conventions 0.66b1 requires opentelemetry-api==1.45.1, but you have opentelemetry-api 1.45.2 which is incompatible.
kfp 2.17.0 requires protobuf<7.0,>=6.31.1, but you have protobuf 7.36.2 which is incompatible.
kfp 2.17.0

### Set Google Cloud project information
Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
%pip install uv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys

!uv pip install --python "{sys.executable}" "mlflow==3.18.0" "dvc[gs]" pandas scikit-learn

Using Python 3.12.15 environment at: /opt/micromamba
Resolved 170 packages in 828ms                                       
error: failed to remove directory
       `/opt/micromamba/lib/python3.12/site-packages/opentelemetry/semconv/__pycache__`:
       Permission denied (os error 13)


In [4]:
import subprocess
import sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "dvc[gs]"],
    check=True,
)

Using Python 3.12.15 environment at: /opt/micromamba
Checked 1 package in 19ms


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/opt/micromamba/bin/python3', 'dvc[gs]'], returncode=0)

In [5]:
subprocess.run(
    [sys.executable, "-m", "dvc", "pull", "iris.csv.dvc"],
    check=True,
)

Everything is up to date.


I1111 22:31:55.977149   25455 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I1111 22:31:55.988048   25455 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


CompletedProcess(args=['/opt/micromamba/bin/python3', '-m', 'dvc', 'pull', 'iris.csv.dvc'], returncode=0)

In [7]:
PROJECT_ID = "project-4c4cfed2-deb7-4275-965"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

### Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.

In [8]:
BUCKET_URI = f"gs://project-4c4cfed2-mlops-course"  # @param {type:"string"}

**If your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [4]:
print(BUCKET_URI)

! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}

gs://project-4c4cfed2-mlops-course
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Creating gs://project-4c4cfed2-mlops-course/...
ServiceException: 409 A Cloud Storage bucket named 'project-4c4cfed2-mlops-course' already exists. Try another name. Bucket names must be globally unique across all Google Cloud projects, including those outside of your organization.


### Initialize Vertex AI SDK for Python

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

In [9]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

### Import the required libraries

In [10]:
import os
import sys

### Configure resource names

Set a name for the following parameters:

`MODEL_ARTIFACT_DIR` - Folder directory path to your model artifacts within a Cloud Storage bucket, for example: "my-models/fraud-detection/trial-4"

`REPOSITORY` - Name of the Artifact Repository to create or use.

`IMAGE` - Name of the container image that is pushed to the repository.

`MODEL_DISPLAY_NAME` - Display name of Vertex AI model resource.

In [11]:
!pip install datetime

In [12]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
MODEL_ARTIFACT_DIR = "my-models/iris-classifier-{timestamp}"  # @param {type:"string"}
REPOSITORY = "iris-classifier-repo"  # @param {type:"string"}
IMAGE = "iris-classifier-img"  # @param {type:"string"}
MODEL_DISPLAY_NAME = "iris-classifier"  # @param {type:"string"}

# Set the defaults if no names were specified
if MODEL_ARTIFACT_DIR == "[your-artifact-directory]":
    MODEL_ARTIFACT_DIR = "custom-container-prediction-model-{timestamp}"

if REPOSITORY == "[your-repository-name]":
    REPOSITORY = "custom-container-prediction"

if IMAGE == "[your-image-name]":
    IMAGE = "sklearn-fastapi-server"

if MODEL_DISPLAY_NAME == "[your-model-display-name]":
    MODEL_DISPLAY_NAME = "sklearn-custom-container"

## Simple Decision Tree model
Build a Decision Tree model on iris data

In [2]:
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_registry_uri("sqlite:///mlflow.db")

data = pd.read_csv("iris.csv")

features = [
    "sepal_length", "sepal_width",
    "petal_length", "petal_width"
]

remaining, test = train_test_split(
    data, test_size=0.2,
    stratify=data["species"], random_state=42
)

train, validation = train_test_split(
    remaining, test_size=0.25,
    stratify=remaining["species"], random_state=42
)

print("Training:", len(train))
print("Validation:", len(validation))
print("Testing:", len(test))

Training: 108
Validation: 36
Testing: 36


In [3]:
max_depth = 1
min_samples_split = 2

mlflow.set_experiment("iris-experiment-1")

with mlflow.start_run():
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=1
    )
    model.fit(train[features], train["species"])

    predictions = model.predict(validation[features])

    accuracy = accuracy_score(validation["species"], predictions)
    precision = precision_score(
        validation["species"], predictions,
        average="weighted", zero_division=0
    )

    mlflow.log_params({
        "max_depth": max_depth,
        "min_samples_split": min_samples_split
    })

    mlflow.log_metrics({
        "validation_accuracy": accuracy,
        "validation_precision": precision
    })

    mlflow.sklearn.log_model(
        model,
        name="model",
        registered_model_name="iris_classifier",
        input_example=train[features].head(),
        serialization_format="cloudpickle"
    )

    print("Validation accuracy:", accuracy)
    print("Validation precision:", precision)

2026/11/11 23:30:02 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/11 23:30:02 INFO mlflow.store.db.utils: Updating database tables
2026/11/11 23:30:06 INFO mlflow.tracking.fluent: Experiment with name 'iris-experiment-1' does not exist. Creating a new experiment.
2026/11/11 23:30:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/11/11 23:30:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Validation accuracy: 0.6666666666666666
Validation precision: 0.5


Successfully registered model 'iris_classifier'.
Created version '1' of model 'iris_classifier'.


In [4]:
max_depth = 3
min_samples_split = 10

mlflow.set_experiment("iris-experiment-2")

with mlflow.start_run():
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=1
    )
    model.fit(train[features], train["species"])

    predictions = model.predict(validation[features])

    accuracy = accuracy_score(validation["species"], predictions)
    precision = precision_score(
        validation["species"], predictions,
        average="weighted", zero_division=0
    )

    mlflow.log_params({
        "max_depth": max_depth,
        "min_samples_split": min_samples_split
    })

    mlflow.log_metrics({
        "validation_accuracy": accuracy,
        "validation_precision": precision
    })

    mlflow.sklearn.log_model(
        model,
        name="model",
        registered_model_name="iris_classifier",
        input_example=train[features].head(),
        serialization_format="cloudpickle"
    )

    print("Validation accuracy:", accuracy)
    print("Validation precision:", precision)

2026/11/11 23:31:24 INFO mlflow.tracking.fluent: Experiment with name 'iris-experiment-2' does not exist. Creating a new experiment.
2026/11/11 23:31:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/11/11 23:31:28 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Validation accuracy: 0.9722222222222222
Validation precision: 0.9743589743589745


Registered model 'iris_classifier' already exists. Creating a new version of this model...
Created version '2' of model 'iris_classifier'.


In [ ]:
max_depth = 1
min_samples_split = 2

mlflow.set_experiment("iris-experiment-1")

with mlflow.start_run():
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=1
    )
    model.fit(train[features], train["species"])

    predictions = model.predict(validation[features])

    accuracy = accuracy_score(validation["species"], predictions)
    precision = precision_score(
        validation["species"], predictions,
        average="weighted", zero_division=0
    )

    mlflow.log_params({
        "max_depth": max_depth,
        "min_samples_split": min_samples_split
    })

    mlflow.log_metrics({
        "validation_accuracy": accuracy,
        "validation_precision": precision
    })

    mlflow.sklearn.log_model(
        model,
        name="model",
        registered_model_name="iris_classifier",
        input_example=train[features].head(),
        serialization_format="cloudpickle"
    )

    print("Validation accuracy:", accuracy)
    print("Validation precision:", precision)

In [5]:
from IPython.display import display, HTML

display(HTML(
    '<a href="/proxy/5000/" target="_blank">'
    'Open MLflow UI</a>'
))

In [6]:
import mlflow.sklearn
from sklearn.metrics import accuracy_score, precision_score

model_version = "2"  

model_uri = f"models:/iris_classifier/{model_version}"
registered_model = mlflow.sklearn.load_model(model_uri)

predictions = registered_model.predict(test[features])

print("Loaded from registry:", model_uri)
print("Test accuracy:", accuracy_score(test["species"], predictions))
print(
    "Test precision:",
    precision_score(
        test["species"], predictions,
        average="weighted", zero_division=0
    )
)

Loaded from registry: models:/iris_classifier/2
Test accuracy: 0.9444444444444444
Test precision: 0.9523809523809523


In [20]:
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

In [21]:
mod_dt = DecisionTreeClassifier(max_depth = 3, random_state = 1)
mod_dt.fit(X_train,y_train)
prediction=mod_dt.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

The accuracy of the Decision Tree is 0.983


In [22]:
import pickle
import joblib

joblib.dump(mod_dt, "artifacts/model.joblib")

['artifacts/model.joblib']

### Upload model artifacts and custom code to Cloud Storage

Before you can deploy your model for serving, Vertex AI needs access to the following files in Cloud Storage:

* `model.joblib` (model artifact)
* `preprocessor.pkl` (model artifact)

Run the following commands to upload your files:

In [23]:
!gsutil cp artifacts/model.joblib {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://artifacts/model.joblib [Content-Type=application/octet-stream]...
/ [1 files][  2.5 KiB/  2.5 KiB]                                                
Operation completed over 1 objects/2.5 KiB.                                      


In [25]:
from google.cloud import storage

In [26]:
from sklearn import metrics

In [27]:
model_filename = "model.joblib"

In [28]:
local_model_path=f"artifacts/{model_filename}"

In [29]:
client = storage.Client(project=PROJECT_ID)

In [31]:
bucket_name = BUCKET_URI.replace("gs://", "")
bucket=client.bucket(bucket_name)
blob = bucket.blob(f"{MODEL_ARTIFACT_DIR}/{model_filename}")

In [32]:
loaded_model = joblib.load(local_model_path)
predictions = loaded_model.predict(X_test)

In [33]:
accuracy = metrics.accuracy_score(y_test, predictions)

In [34]:
print(accuracy)


0.9833333333333333


In [2]:
!git reset --soft HEAD~1
!git reset SDK_Custom_Container_Prediction.ipynb
!git add SDK_Custom_Container_Prediction.ipynb
!git commit -m "Add MLOps custom container prediction notebook for week_1"

[week_1 d62eb98] Add MLOps custom container prediction notebook for week_1
 1 file changed, 1135 insertions(+)
 create mode 100644 SDK_Custom_Container_Prediction.ipynb


In [3]:

!git push -u origin week_1

Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (5/5), 8.27 KiB | 4.14 MiB/s, done.
Total 5 (delta 0), reused 0 (delta 0), pack-reused 0
remote: error: GH013: Repository rule violations found for refs/heads/week_1.
remote: 
remote: - GITHUB PUSH PROTECTION
remote:   —————————————————————————————————————————
remote:     Resolve the following violations before pushing again
remote: 
remote:     - Push cannot contain secrets
remote: 
remote:     
remote:      (?) Learn how to resolve a blocked push
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push
remote:     
remote:     
remote:       —— GitHub Personal Access Token ——————————————————————
remote:        locations:
remote:          - commit: d62eb98c757d4e0d5a9a868de7bfc549610a9d38

In [7]:
!git checkout -b week_1
!git add SDK_Custom_Container_Prediction.ipynb
!git commit -m "Add MLOps custom container prediction notebook for week_1"
!git push -u origin week_1

Switched to a new branch 'week_1'
[week_1 3ac8b17] Add MLOps custom container prediction notebook for week_1
 1 file changed, 1074 insertions(+)
 create mode 100644 SDK_Custom_Container_Prediction.ipynb
Username for 'https://github.com': ^C


In [9]:
import pandas as pd

data = pd.read_csv("iris.csv")

print("Rows:", len(data))
print("Columns:", data.columns.tolist())
print(data["species"].value_counts())
data.head()

Rows: 180
Columns: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
species
setosa        60
versicolor    60
virginica     60
Name: count, dtype: int64


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


ImportError: cannot import name 'ReadableSpan' from 'opentelemetry.sdk.trace' (unknown location)